In [ ]:
# @title Descargar repositorio

!git clone --recurse-submodules https://github.com/ServeurpersoCom/yue2.cpp
%cd yue2.cpp

!wget https://github.com/Harzur/YuE2-Q8.0-Colab-T4/releases/download/v1.0-t4/yue2_t4_binaries.zip -O /content/yue2_t4_binaries.zip

import os
import zipfile

ruta_zip = "/content/yue2_t4_binaries.zip"
carpeta_en_zip = "build/"
destino = "/content/yue2.cpp/build"

os.makedirs(destino, exist_ok=True)

with zipfile.ZipFile(ruta_zip, "r") as z:
  for file_info in z.infolist():
    if file_info.filename.startswith(
        carpeta_en_zip
    ) and not file_info.is_dir():

      archivo_relativo = file_info.filename[len(carpeta_en_zip) :]

      if not archivo_relativo:
        continue

      ruta_salida = os.path.join(destino, archivo_relativo)
      os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)

      with z.open(file_info) as fuente, open(ruta_salida, "wb") as salida:
        salida.write(fuente.read())


In [ ]:
# @title Descargar modelos

%cd /content/yue2.cpp
!pip install -q -U huggingface_hub --no-warn-conflicts
!chmod +x models.sh
!./models.sh

In [ ]:
# @title Generar enlace Cloudflared para abrir la WebUI en tu navegador
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

%cd /content/yue2.cpp
import subprocess, time, re, os

# Dar permisos
!chmod -R +x ./build/
!chmod +x ./cloudflared-linux-amd64


# Iniciar Cloudflare redirigiendo la salida a un archivo de registro
log_file = open("tunnel.log", "w")
tunnel = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8087"],
                          stdout=log_file, stderr=log_file)
# Esperar a que aparezca la URL real (la que empieza por https://)
url = None
print("Generando enlace seguro...")
for _ in range(15):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if match:
                url = match.group(0)
                break
if url:
    print(f"\n" + "="*60)
    print(f"👉 HAZ CLIC AQUÍ PARA ABRIR LA WEBUI: {url}")
    print("="*60 + "\n")
else:
    print("\n⚠️ No se detectó a tiempo, pero el servidor iniciará igualmente.")

# 4. Iniciar el servidor con el Transcriptor (SheetSage2) y el Tokenizador activados
!./build/yue-server --host 0.0.0.0 --port 8087 \
    --model models/YuE2-3B-Q8_0.gguf \
    --vae models/YuE2-Vae-F32.gguf \
    --transcriber models/SheetSage2-Q8_0.gguf \
    --tokenizer models/yue2-mothersuperior-realaudio-tokenizer-v4-Q8_0.gguf > server.log 2>&1